In [3]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

# Text pre-processing

1. Convert to lowercase
2. Remove URLs (http, https)
3. Remove HTML
4. Remove Punctuation
5. Remove stopwords
6. Stemming
7. Encoding Sentiments
8. Vectorization (TF-IDF)

## Converting to lowercase

In [7]:
df["review"] = df["review"].str.lower()

## Removing URLs

In [8]:
# demo
import re # regex

sample_text = "abc is the word, abc" # abc -> xyz

new_text = re.sub("abc", "xyz", sample_text)
new_text

'xyz is the word, xyz'

In [9]:
def remove_urls(text):
    return re.sub( r"http\S+" , "", text) # \S is non-whitespace character
    # /s is whitespace character

df["review"] = df["review"].apply(remove_urls)

## Removing HTML

In [10]:
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df["review"] = df["review"].apply(remove_html)

## Remove Punctuations

In [11]:
def remove_punctuations(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

df["review"] = df["review"].apply(remove_punctuations)

In [12]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


## Remove stopwords
We need to do tokenization before this step.

In [13]:
import nltk # natural language tool kit
nltk.download("punkt") # punkt is the official tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sidharthgupta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sidharthgupta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sidharthgupta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)

    filtered_tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [17]:
print(df.iloc[0]["review"])

one reviewers mentioned watching 1 oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordit called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy high agenda em city home manyaryans muslims gangstas latinos christians italians irish moreso scuffles death stares dodgy dealings shady agreements never far awayi would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high levels graphic violence violence injustice crooked guards wholl sold nickel inmates wholl kill order get away well mannered middle c

## Stemming

In [18]:
from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_words = []
    
    for word in tokens:
        stemmed_token = ps.stem(word)
        stemmed_words.append(stemmed_token)

    text = " ".join(stemmed_words)
    return text

In [20]:
df["review"] = df["review"].apply(stemming)

In [21]:
print(df.iloc[0]["review"])

one review mention watch 1 oz episod youll hook right exactli happen meth first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc hardcor classic use wordit call oz nicknam given oswald maximum secur state penitentari focus mainli emerald citi experiment section prison cell glass front face inward privaci high agenda em citi home manyaryan muslim gangsta latino christian italian irish moreso scuffl death stare dodgi deal shadi agreement never far awayi would say main appeal show due fact goe show wouldnt dare forget pretti pictur paint mainstream audienc forget charm forget romanceoz doesnt mess around first episod ever saw struck nasti surreal couldnt say readi watch develop tast oz got accustom high level graphic violenc violenc injustic crook guard wholl sold nickel inmat wholl kill order get away well manner middl class inmat turn prison bitch due lack street skill prison experi watch oz may becom co

## Encoding

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

## Vectorization

*TF-IDF*  
**Mistake** - Then sequence/order of words don't matter.

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=10000)

X = tf.fit_transform(df["review"])

In [24]:
X 

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4318375 stored elements and shape (49582, 10000)>

In [25]:
y = df["sentiment"].to_numpy()

In [26]:
type(y)

numpy.ndarray

# Dataset and DataLoader

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
X_train.shape

(39665, 10000)

In [29]:
y_train.shape

(39665,)

In [30]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [31]:
X_train = X_train.toarray()
X_test = X_test.toarray()

# Earlier, it was sparse matrix. I converted them to numpy array.

In [32]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float()
)

In [33]:
train_loader = DataLoader(train_set, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

# Build RNN

In [34]:
import torch.nn as nn
import torch.optim as optim

In [41]:
import numpy as np
np.unique(y)

array([0, 1])

In [ ]:
# class RNN(nn.Module):
#     def __init__(self, input_size, hidden_size = 128, num_layers = 1):
#         super().__init__()

#         self.hidden_size = hidden_size
#         self.num_layers = num_layers 

#         # RNN layer
#         self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

#         # Fully Connectede Layer
#         self.fc = nn.Linear(hidden_size=128)

#     def forward(self, x):
        